# Neurosymbolic integration — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def sigmoid(z): return 1/(1+np.exp(-z))
def bce(p,y):
    p=np.clip(np.asarray(p,dtype=float),1e-9,1-1e-9)
    y=np.asarray(y,dtype=float)
    return -(y*np.log(p)+(1-y)*np.log(1-p))
print('setup ok')

## Combining a neural score with a symbolic constraint
Neurosymbolic integration makes a model obey a rule by adding a differentiable constraint penalty to an ordinary neural loss. These five examples show fuzzy truth values, differentiable AND/OR surfaces, implication satisfaction, and how the combined objective changes as the rule weight grows.

In [ ]:
# 1) fuzzy truth turns neural probabilities into logic values in [0,1]
p=np.array([0.20,0.70,0.90]); y=np.array([0,1,1])
loss=bce(p,y); mean_loss=float(loss.mean())
plt.figure(figsize=(6,3)); plt.bar(['x1','x2','x3'],p,color='tab:blue'); plt.ylim(0,1); plt.ylabel('truth / probability'); plt.title(f'fuzzy truth values, BCE mean={mean_loss:.4f}')
assert abs(mean_loss-0.22839300363692283)<1e-12

In [ ]:
# 2) product t-norm AND is differentiable: A AND B = A*B
A=np.linspace(0,1,60); B=np.linspace(0,1,60); AA,BB=np.meshgrid(A,B); AND=AA*BB
plt.figure(figsize=(5,4)); plt.contourf(AA,BB,AND,levels=20,cmap='viridis'); plt.colorbar(label='A AND B'); plt.xlabel('A'); plt.ylabel('B'); plt.title('differentiable AND surface')
assert abs(float(0.7*0.6)-0.42)<1e-12

In [ ]:
# 3) probabilistic OR is differentiable: A OR B = A + B - A*B
OR=AA+BB-AA*BB
plt.figure(figsize=(5,4)); plt.contourf(AA,BB,OR,levels=20,cmap='magma'); plt.colorbar(label='A OR B'); plt.xlabel('A'); plt.ylabel('B'); plt.title('differentiable OR surface')
assert abs(float(0.7+0.6-0.7*0.6)-0.88)<1e-12

In [ ]:
# 4) soft implication A -> B is satisfied by max(1-A, B)
Avals=np.array([0.2,0.7,0.9]); Bvals=np.array([0.8,0.6,0.4]); sat=np.maximum(1-Avals,Bvals); penalty=1-sat
plt.figure(figsize=(6,3)); plt.bar(np.arange(3)-0.15,sat,width=0.3,label='satisfaction'); plt.bar(np.arange(3)+0.15,penalty,width=0.3,label='penalty'); plt.ylim(0,1); plt.xticks(range(3),['case1','case2','case3']); plt.legend(); plt.title('violations become penalties')
assert np.allclose(sat,np.array([0.8,0.6,0.4]))

In [ ]:
# 5) the combined objective trades data fit against rule satisfaction
lambdas=np.array([0,0.5,1,2,5],dtype=float); neural_loss=0.22839300363692283; rule_penalty=0.4; total=neural_loss+lambdas*rule_penalty
plt.figure(figsize=(6,3)); plt.plot(lambdas,total,'-o'); plt.xlabel('constraint weight lambda'); plt.ylabel('total loss'); plt.title('larger lambda makes rule violation costly')
assert abs(float(total[3])-1.0283930036369228)<1e-12